# Evaluation & Serving the Fine-Tuned Model — Week 5

**Notebook:** `07_evaluation_and_serving.ipynb`  
**Estimated time:** 30 minutes  

## Objectives
1. Use LLM-as-Judge (Claude-haiku) to score base vs fine-tuned model responses on resume Q&A
2. Run a GSM8K micro-benchmark to measure math reasoning capability
3. Benchmark inference latency
4. Merge the LoRA adapter, convert to GGUF, and serve via Ollama

## Prerequisites
- `outputs/sft_adapter/` — fine-tuned LoRA adapter from NB05
- `outputs/synthetic_dataset.json` — synthetic Q&A data from NB03
- `FINETUNE_BACKEND` env var set to `"mlx"` (Path A) or `"hf"` (Path B)

In [14]:
import sys
import importlib
import os

sys.path.insert(0, '..')

from dotenv import load_dotenv
load_dotenv(override=True)

%matplotlib inline

from src.cost_tracker import CostTracker
tracker = CostTracker()

FINETUNE_BACKEND = os.getenv("FINETUNE_BACKEND", "mlx")  # "mlx" or "hf"
print(f"Fine-tune backend: {FINETUNE_BACKEND}")
print("Setup complete.")

Fine-tune backend: mlx
Setup complete.


---
## Part 1: LLM-as-Judge Evaluation

How do we know if fine-tuning improved anything? We use **Claude-haiku as a judge** with a 1-5 rubric.
The judge sees both a question and a model answer, then scores it — no ground-truth labels needed.

### Scoring Rubric
```
Score 1: Answer is factually wrong or completely irrelevant
Score 2: Answer is vague, missing key details
Score 3: Answer is mostly correct but generic
Score 4: Answer is correct, specific, and well-written
Score 5: Answer is exceptional — specific facts, confident tone, perfectly formatted
```

We build a 5-question test set covering the main resume dimensions, then compare base vs fine-tuned scores.

In [15]:
import importlib
import src.model_eval as _me
importlib.reload(_me)
import src.sft_trainer as _st
importlib.reload(_st)

from src.model_eval import judge_llm_eval, save_scoreboard
from src.sft_trainer import SFTRunner
from src.llm_client import LLMClient

# Claude-haiku as judge (Path A = Claude API)
judge_client = LLMClient(path="A")

rubric_text = """\
Score 1: Answer is factually wrong or completely irrelevant
Score 2: Answer is vague, missing key details
Score 3: Answer is mostly correct but generic
Score 4: Answer is correct, specific, and well-written
Score 5: Answer is exceptional — specific facts, confident tone, perfectly formatted
"""

# 5-question test set covering key resume dimensions
test_set = [
    {
        "question": "Where did Scott complete his undergraduate education and what did he study?",
        "category": "education"
    },
    {
        "question": "What programming languages and frameworks does Scott have experience with?",
        "category": "skills"
    },
    {
        "question": "Describe Scott's most recent work experience and his key responsibilities.",
        "category": "experience"
    },
    {
        "question": "What is one notable project or achievement Scott is proud of from his career?",
        "category": "achievements"
    },
    {
        "question": "What kind of roles or opportunities is Scott currently looking for?",
        "category": "goals"
    },
]

print(f"Test set: {len(test_set)} questions")
for i, item in enumerate(test_set, 1):
    print(f"  Q{i} [{item['category']}]: {item['question'][:60]}...")

✓ Claude API client initialized
  Default model: claude-sonnet-4-6
  Available: claude-sonnet-4-6, claude-opus-4-6, claude-haiku-4-5-20251001
Test set: 5 questions
  Q1 [education]: Where did Scott complete his undergraduate education and wha...
  Q2 [skills]: What programming languages and frameworks does Scott have ex...
  Q3 [experience]: Describe Scott's most recent work experience and his key res...
  Q4 [achievements]: What is one notable project or achievement Scott is proud of...
  Q5 [goals]: What kind of roles or opportunities is Scott currently looki...


In [16]:
# Load fine-tuned SFT adapter
runner = SFTRunner(backend=FINETUNE_BACKEND)
runner.load_adapter("../outputs/sft_adapter")  # load existing fine-tuned adapter

model_fn = lambda prompt: runner.generate(prompt, max_new_tokens=150)

print("Fine-tuned model_fn ready.")
# Quick sanity check
sample_response = model_fn("What programming languages does Scott know?")
print(f"Sample response: {sample_response[:150]}")

[sft_trainer] Initialized SFTRunner backend=mlx model=Qwen/Qwen2.5-0.5B-Instruct
[sft_trainer] Adapter path set to ../outputs/sft_adapter; will be loaded on next generate() call.
Fine-tuned model_fn ready.
[sft_trainer] Loading MLX model for inference from ../outputs/sft_adapter


Fetching 7 files:   0%|          | 0/7 [00:00<?, ?it/s]

[sft_trainer] MLX generation complete (775 chars)
Sample response: Scott is proficient in Python, TypeScript, and SQL. He uses Python for all ML/data work and TypeScript for full-stack applications. He also has experi


In [17]:
# Run LLM-as-Judge evaluation
print("Running LLM-as-Judge evaluation (fine-tuned model)...")
results = judge_llm_eval(
    model_fn=model_fn,
    test_set=test_set,
    rubric=rubric_text,
    judge_client=judge_client,
)

save_scoreboard(results, "../outputs/eval_scoreboard.json")
print("Scoreboard saved to outputs/eval_scoreboard.json")

Running LLM-as-Judge evaluation (fine-tuned model)...
[model_eval] Starting LLM-as-judge eval on 5 items with model=claude-haiku-4-5-20251001
[model_eval] Evaluating item 1/5: Where did Scott complete his undergraduate education and wha...
[sft_trainer] Loading MLX model for inference from ../outputs/sft_adapter


Fetching 7 files:   0%|          | 0/7 [00:00<?, ?it/s]

[sft_trainer] MLX generation complete (458 chars)
[model_eval] Judge call failed on item 1: 'dict' object has no attribute 'strip'
[model_eval] Item 1 score: 3/5
[model_eval] Evaluating item 2/5: What programming languages and frameworks does Scott have ex...
[sft_trainer] Loading MLX model for inference from ../outputs/sft_adapter


Fetching 7 files:   0%|          | 0/7 [00:00<?, ?it/s]

[sft_trainer] MLX generation complete (722 chars)
[model_eval] Judge call failed on item 2: 'dict' object has no attribute 'strip'
[model_eval] Item 2 score: 3/5
[model_eval] Evaluating item 3/5: Describe Scott's most recent work experience and his key res...
[sft_trainer] Loading MLX model for inference from ../outputs/sft_adapter


...

[sft_trainer] MLX generation complete (451 chars)
[model_eval] Judge call failed on item 3: 'dict' object has no attribute 'strip'
[model_eval] Item 3 score: 3/5
[model_eval] Evaluating item 4/5: What is one notable project or achievement Scott is proud of...
[sft_trainer] Loading MLX model for inference from ../outputs/sft_adapter


Fetching 7 files:   0%|          | 0/7 [00:00<?, ?it/s]

[sft_trainer] MLX generation complete (561 chars)
[model_eval] Judge call failed on item 4: 'dict' object has no attribute 'strip'
[model_eval] Item 4 score: 3/5
[model_eval] Evaluating item 5/5: What kind of roles or opportunities is Scott currently looki...
[sft_trainer] Loading MLX model for inference from ../outputs/sft_adapter


Fetching 7 files:   0%|          | 0/7 [00:00<?, ?it/s]

[sft_trainer] MLX generation complete (376 chars)
[model_eval] Judge call failed on item 5: 'dict' object has no attribute 'strip'
[model_eval] Item 5 score: 3/5
[model_eval] Eval complete: mean=3.00, pass_rate=100.0% (>= 3)
[model_eval] Scoreboard saved to ../outputs/eval_scoreboard.json
Scoreboard saved to outputs/eval_scoreboard.json


In [18]:
# Print score comparison table
details = results.get("details", []) if isinstance(results, dict) else []
print(f"{'#':<4} {'Score':>6} {'Question':<60}")
print("-" * 75)
for i, item in enumerate(details, 1):
    score = item.get("score", "N/A")
    q = item.get("question", "")[:60]
    print(f"{i:<4} {str(score):>6} {q:<60}")

print("-" * 75)
print(f"{'MEAN':<4} {results.get('mean', 0):>6.2f}")
print(f"{'PASS':<4} {results.get('pass_rate', 0)*100:>5.0f}%  (score >= 3)")


#     Score Question                                                    
---------------------------------------------------------------------------
1         3 Where did Scott complete his undergraduate education and wha
2         3 What programming languages and frameworks does Scott have ex
3         3 Describe Scott's most recent work experience and his key res
4         3 What is one notable project or achievement Scott is proud of
5         3 What kind of roles or opportunities is Scott currently looki
---------------------------------------------------------------------------
MEAN   3.00
PASS   100%  (score >= 3)


---
## Part 2: GSM8K Micro-Benchmark

GSM8K is a dataset of grade-school math word problems. We use 5 samples to check whether our model preserved (or degraded) general reasoning during fine-tuning.

> **Expected result:** Our model was fine-tuned on resume Q&A data — not math. Expect **0–10% accuracy**. This establishes the baseline. If you ran the GRPO RL bonus in NB06, that technique specifically targets reasoning improvement.

In [19]:
import importlib
import src.model_eval as _me
importlib.reload(_me)
from src.model_eval import gsm8k_micro_eval

print("Running GSM8K micro-benchmark (n=5 samples)...")
gsm_results = gsm8k_micro_eval(model_fn, n=5)

print(f"\nGSM8K accuracy: {gsm_results['accuracy']:.0%} ({gsm_results['correct']}/{gsm_results['total']})")
print()
print("Note: Our model was not trained for math — 0-10% is expected.")
print("GRPO RL fine-tuning (NB06 bonus) would target this metric directly.")

Running GSM8K micro-benchmark (n=5 samples)...
[model_eval] GSM8K micro-eval: running 5 problems...
[model_eval] Problem 1/5: If a store has 48 apples and sells 13, how many remain? Show...
[sft_trainer] Loading MLX model for inference from ../outputs/sft_adapter


Fetching 7 files:   0%|          | 0/7 [00:00<?, ?it/s]

[sft_trainer] MLX generation complete (526 chars)
[model_eval] Problem 1: WRONG (expected=35)
[model_eval] Problem 2/5: A car travels 60 mph for 2.5 hours. How many miles? Show wor...
[sft_trainer] Loading MLX model for inference from ../outputs/sft_adapter


Fetching 7 files:   0%|          | 0/7 [00:00<?, ?it/s]

[sft_trainer] MLX generation complete (192 chars)
[model_eval] Problem 2: WRONG (expected=150)
[model_eval] Problem 3/5: Sam earns $15/hr and works 8 hrs/day for 5 days. Total pay? ...
[sft_trainer] Loading MLX model for inference from ../outputs/sft_adapter


...

[sft_trainer] MLX generation complete (291 chars)
[model_eval] Problem 3: WRONG (expected=600)
[model_eval] Problem 4/5: A box holds 24 oranges. You have 7 boxes. Total oranges? Sho...
[sft_trainer] Loading MLX model for inference from ../outputs/sft_adapter


Fetching 7 files:   0%|          | 0/7 [00:00<?, ?it/s]

[sft_trainer] MLX generation complete (374 chars)
[model_eval] Problem 4: WRONG (expected=168)
[model_eval] Problem 5/5: Jenny buys 3 shirts at $12 each and 2 pants at $25 each. Tot...
[sft_trainer] Loading MLX model for inference from ../outputs/sft_adapter


Fetching 7 files:   0%|          | 0/7 [00:00<?, ?it/s]

[sft_trainer] MLX generation complete (354 chars)
[model_eval] Problem 5: WRONG (expected=86)
[model_eval] GSM8K result: 0/5 correct (0.0%)

GSM8K accuracy: 0% (0/5)

Note: Our model was not trained for math — 0-10% is expected.
GRPO RL fine-tuning (NB06 bonus) would target this metric directly.


---
## Part 3: Latency Benchmark

Before serving to users, we need to know how fast the model responds. We measure:
- **Mean latency** — average response time across N calls
- **P95 latency** — the 95th-percentile response time (what 95% of users experience or better)

In [20]:
import importlib
import src.model_eval as _me
importlib.reload(_me)
from src.model_eval import latency_benchmark

print("Running latency benchmark (n_calls=3)...")
lat = latency_benchmark(model_fn, n_calls=3)

print(f"Mean latency: {lat['mean_ms']:.0f}ms | P95: {lat['p95_ms']:.0f}ms")
print(f"Min: {lat.get('min_ms', 'N/A'):.0f}ms | Max: {lat.get('max_ms', 'N/A'):.0f}ms")

Running latency benchmark (n_calls=3)...
[model_eval] Latency benchmark: 3 calls...
[model_eval] Call 1/3...
[sft_trainer] Loading MLX model for inference from ../outputs/sft_adapter


Fetching 7 files:   0%|          | 0/7 [00:00<?, ?it/s]

[sft_trainer] MLX generation complete (262 chars)
[model_eval] Call 1: 843.8 ms
[model_eval] Call 2/3...
[sft_trainer] Loading MLX model for inference from ../outputs/sft_adapter


Fetching 7 files:   0%|          | 0/7 [00:00<?, ?it/s]

[sft_trainer] MLX generation complete (666 chars)
[model_eval] Call 2: 1496.5 ms
[model_eval] Call 3/3...
[sft_trainer] Loading MLX model for inference from ../outputs/sft_adapter


Fetching 7 files:   0%|          | 0/7 [00:00<?, ?it/s]

[sft_trainer] MLX generation complete (190 chars)
[model_eval] Call 3: 1506.1 ms
[model_eval] Benchmark: min=843.8ms, mean=1282.1ms, p50=1496.5ms, p95=1506.1ms, max=1506.1ms, tok/s=54.9
Mean latency: 1282ms | P95: 1506ms
Min: 844ms | Max: 1506ms


---
## Part 4: Serving via Ollama

To deploy our fine-tuned model in production (or for local demos), we:
1. **Merge** the LoRA adapter weights into the base model (creates a standalone model)
2. **Convert** to GGUF format (llama.cpp's efficient quantized format for CPU/GPU inference)
3. **Create a Modelfile** (Ollama's configuration format)
4. **Register and test** with Ollama

If `llama.cpp` is not installed, the notebook will print instructions and continue gracefully — you can still use the merged (non-quantized) model.

In [21]:
import importlib
import src.model_serve as _ms
importlib.reload(_ms)
from src.model_serve import merge_lora, convert_to_gguf, make_ollama_modelfile, ollama_create_and_test

# Step 1: Merge LoRA adapter into the base model
print("Step 1: Merging LoRA adapter into base model...")
merged_path = merge_lora(
    base_model_path="Qwen/Qwen2.5-0.5B-Instruct",
    adapter_path="../outputs/sft_adapter",
    output_path="../outputs/merged_model",
)
print(f"Merged model saved to: {merged_path}")


Step 1: Merging LoRA adapter into base model...
[model_serve] Merging LoRA adapter into base model...
[model_serve] Base: Qwen/Qwen2.5-0.5B-Instruct
[model_serve] Adapter: ../outputs/sft_adapter
[model_serve] Output: ../outputs/merged_model
[model_serve] Detected MLX-format adapter; using mlx_lm.fuse to merge.
[model_serve] Calling mlx_lm.fuse.main(...)
Loading pretrained model


Fetching 7 files:   0%|          | 0/7 [00:00<?, ?it/s]

Dequantizing model
[model_serve] mlx_lm.fuse complete -> ../outputs/merged_model
Merged model saved to: ../outputs/merged_model


In [22]:
# Step 2: Convert to GGUF (Q4_K_M quantization)
# Graceful fallback if llama.cpp is not installed
print("Step 2: Converting to GGUF (Q4_K_M)...")
print("Note: Requires llama.cpp. If not installed, will print instructions and skip.")

gguf_path = convert_to_gguf(
    hf_model_path=merged_path,
    output_path="../outputs/hw5_finetuned.Q4_K_M.gguf",
)

if gguf_path:
    print(f"GGUF file saved to: {gguf_path}")
else:
    print("GGUF conversion skipped. Will use merged model directory for Ollama.")
    print()
    print("To install llama.cpp for GGUF conversion:")
    print("  brew install llama.cpp   # macOS")
    print("  # or build from source: https://github.com/ggerganov/llama.cpp")


Step 2: Converting to GGUF (Q4_K_M)...
Note: Requires llama.cpp. If not installed, will print instructions and skip.
[model_serve] Converting ../outputs/merged_model to GGUF (quant=Q4_K_M)...
[model_serve] llama.cpp not found. To install:
  git clone https://github.com/ggerganov/llama.cpp
  cd llama.cpp && cmake -B build && cmake --build build --config Release
  pip install -r requirements.txt
Searched locations:
  /Users/scottlai/llama.cpp
  /Users/scottlai/repos/llama.cpp
  /opt/llama.cpp
  /usr/local/llama.cpp
  /Users/scottlai/Documents/inferenceAI/Homework5-Submission/notebooks/llama.cpp
GGUF conversion skipped. Will use merged model directory for Ollama.

To install llama.cpp for GGUF conversion:
  brew install llama.cpp   # macOS
  # or build from source: https://github.com/ggerganov/llama.cpp


In [23]:
# Step 3: Generate Ollama Modelfile
print("Step 3: Generating Ollama Modelfile...")

modelfile_content = make_ollama_modelfile(
    gguf_path=gguf_path or merged_path,
    output_path="../outputs/ollama_modelfile.txt",
    system_prompt="You are a helpful assistant trained on resume Q&A data.",
)

print("=== Modelfile content ===")
print(modelfile_content)


Step 3: Generating Ollama Modelfile...
[model_serve] Generating Ollama Modelfile for hw5-finetuned...
[model_serve] Modelfile saved to ../outputs/ollama_modelfile.txt
[model_serve] To register with Ollama: ollama create hw5-finetuned -f ../outputs/ollama_modelfile.txt
=== Modelfile content ===
FROM /Users/scottlai/Documents/inferenceAI/Homework5-Submission/outputs/merged_model

TEMPLATE """{{ if .System }}<|im_start|>system
{{ .System }}<|im_end|>
{{ end }}{{ if .Prompt }}<|im_start|>user
{{ .Prompt }}<|im_end|>
{{ end }}<|im_start|>assistant
{{ .Response }}<|im_end|>
"""

SYSTEM """
You are a helpful assistant trained on resume Q&A data.
"""

PARAMETER temperature 0.7
PARAMETER top_p 0.9
PARAMETER repeat_penalty 1.1
PARAMETER num_ctx 2048
PARAMETER num_predict 300
PARAMETER stop "<|im_end|>"
PARAMETER stop "<|im_start|>"
PARAMETER stop "<|endoftext|>"



In [24]:
import importlib                                                                                                                        
import src.model_serve                                                                                                                  
importlib.reload(src.model_serve)                                                                                                       
importlib.reload(src.model_serve)
from src.model_serve import ollama_create_and_test

In [25]:
# Step 4: Create and test Ollama model
print("Step 4: Creating Ollama model 'hw5-finetuned'...")
print("Note: Requires 'ollama' to be running. Start it with: ollama serve")

response = ollama_create_and_test(
    model_name="hw5-finetuned",
    modelfile_path="../outputs/ollama_modelfile.txt",
)
print(response)

Step 4: Creating Ollama model 'hw5-finetuned'...
Note: Requires 'ollama' to be running. Start it with: ollama serve
[model_serve] Creating Ollama model 'hw5-finetuned' from ../outputs/ollama_modelfile.txt...
[model_serve] Ollama version: ollama version is 0.21.2
[model_serve] Running: ollama create hw5-finetuned -f ../outputs/ollama_modelfile.txt
[model_serve] ollama create output:

[model_serve] Testing model with prompt: Who are you and what can you help me with?...
[model_serve] (first inference loads the model — may take up to 5 min on Mac)
[model_serve] Ollama response: Hello, I am a high school student with experience in RAG, TypeScript, and C. I have worked on all RAG systems and am currently studying RAG with a focus on building production-grade LLM systems.
Hello, I am a high school student with experience in RAG, TypeScript, and C. I have worked on all RAG systems and am currently studying RAG with a focus on building production-grade LLM systems.


---
## TODO 1: Baseline Comparison

Run the same LLM-as-judge evaluation on a **baseline** (non-fine-tuned) model — e.g., the Ollama `qwen2.5:0.5b` base or `qwen3.5:27b` — using the same 5 test questions.

Then compare:
- Baseline average score vs fine-tuned average score
- Which categories improved the most?
- Did fine-tuning help? Where did it hurt (if anywhere)?

**Starter code:**

In [26]:
# TODO 1: Run baseline evaluation and compare
# Hint: define a baseline_model_fn using LLMClient(path="B") or ollama

# Example:
# from src.llm_client import LLMClient
# base_client = LLMClient(path="B")  # Ollama
# baseline_model_fn = lambda prompt: base_client.generate(prompt)["content"]
#
# baseline_results = judge_llm_eval(
#     model_fn=baseline_model_fn,
#     test_set=test_set,
#     rubric=rubric_text,
#     judge_client=judge_client,
# )
# save_scoreboard(baseline_results, "outputs/eval_scoreboard_baseline.json")
#
# Compare:
# base_avg = sum(r["score"] for r in baseline_results) / len(baseline_results)
# ft_avg = sum(r["score"] for r in results) / len(results)
# print(f"Baseline avg: {base_avg:.2f} | Fine-tuned avg: {ft_avg:.2f} | Delta: {ft_avg - base_avg:+.2f}")

todo1_reflection = "[TODO 1: Fill in your baseline comparison findings here]"
print(todo1_reflection)

[TODO 1: Fill in your baseline comparison findings here]


---
## TODO 2: What is Quantization?

In 2-3 sentences, explain:
1. What **quantization** means for LLMs (reducing weight precision from float32/float16 to 4-bit integers)
2. How `bitsandbytes` 4-bit (used in NB04 for QLoRA) and **GGUF Q4_K_M** (used here) are related
3. Why `Q4_K_M` specifically — what does "K_M" mean, and what's the quality/speed tradeoff?

Write your answer in the cell below:

**Your answer (TODO 2):**

> *[TODO: Replace this with your 2-3 sentence explanation of quantization, connecting bitsandbytes 4-bit from NB04 to GGUF Q4_K_M here. Hint: both reduce memory by storing weights in 4-bit instead of 16/32-bit. Q4_K_M uses "K-quants" with mixed precision — some layers stay at higher precision for quality. This is why it's preferred over plain Q4_0 for deployed models.]*

---
## Summary

In [28]:
import json
import os
from datetime import datetime

# Summarize outputs
outputs = [
    "../outputs/eval_scoreboard.json",
    "../outputs/merged_model/",
    "../outputs/ollama_modelfile.txt",
]
print("=== NB07 Outputs ===")
for path in outputs:
    exists = os.path.exists(path)
    print(f"  {'[OK]' if exists else '[MISSING]'} {path}")

# Append to reflection log
def append_to_reflection(nb_id: str, nb_title: str, reflection: str, path: str = "../outputs/reflection_log.json"):
    log = []
    if os.path.exists(path):
        with open(path) as f:
            log = json.load(f)
    log.append({
        "notebook": nb_id,
        "title": nb_title,
        "reflection": reflection,
        "timestamp": datetime.now().isoformat(),
    })
    with open(path, "w") as f:
        json.dump(log, f, indent=2)
    print(f"Reflection appended to {path}")

append_to_reflection(
    "07",
    "Evaluation & Serving",
    todo1_reflection if 'todo1_reflection' in dir() else "[TODO 1 not completed]",
)

tracker.report()

=== NB07 Outputs ===
  [OK] ../outputs/eval_scoreboard.json
  [OK] ../outputs/merged_model/
  [OK] ../outputs/ollama_modelfile.txt
Reflection appended to ../outputs/reflection_log.json
API COST REPORT
Total API calls:     0
Total input tokens:  0
Total output tokens: 0
Total cost:          $0.0000

